**Age Model Architecture**

input: image

output: age

regression model

Dataset: UTKFace
Link to download:

part 1: https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669

part 2: https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426


part 3: https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395

# Get the Data

In [ ]:
!rm -rf datasets/

In [ ]:
# Download and unzip the datasets from UTKface

import tarfile
import urllib.request
import os

download_url_part1 = "https://drive.usercontent.google.com/download?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW&export=download&authuser=0&confirm=t&uuid=ed8e8e51-cb3c-4262-8314-d96bb87781d0&at=APcXIO0mPgJPDxNgg9CQHXWo3U2S:1772484352669"
download_url_part2 = "https://drive.usercontent.google.com/download?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b&export=download&authuser=0&confirm=t&uuid=95a6db6e-c371-4d68-8bd1-f39a6f0eb339&at=APcXIO1iISpyVwxBCULNYhXmO8zf:1772485135426"
download_url_part3 = "https://drive.usercontent.google.com/download?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b&export=download&authuser=0&confirm=t&uuid=f8223bbf-436b-4eb6-b67c-914e3f714db2&at=APcXIO1CzF8SjAoIUitQd7krPgcI:1772485183395"

# Download the datasets
download_urls = [download_url_part1, download_url_part2, download_url_part3]
face_path = os.path.join("datasets", "faces")
os.makedirs(face_path, exist_ok=True)

for dataset_num in range(len(download_urls)):
  dataset_folder_name = "part" + str(dataset_num + 1)
  if (not os.path.exists(os.path.join(face_path, dataset_folder_name)) and not os.path.exists(os.path.join(face_path, "images"))):
    print(f"Extracting dataset {dataset_num+1}...")
    faces_tar_location = os.path.join(face_path, "faces" + str(dataset_num+1) + ".tgz")
    urllib.request.urlretrieve(download_urls[dataset_num], faces_tar_location)

    # extract tar dataset
    faces_tgz = tarfile.open(faces_tar_location)
    faces_tgz.extractall(path="datasets/faces", filter="fully_trusted")
    faces_tgz.close()

    # remove tar file
    os.remove(faces_tar_location)

  else:
    print(f"dataset {dataset_num+1} already exists")



In [ ]:
# Combine all data into one folder
import shutil

faces_images_path = os.path.join(face_path, "images")
os.makedirs(faces_images_path, exist_ok=True)

# Get source folders to copy image data from
source_folders = []
for dataset in range(len(download_urls)):
  dataset_path = os.path.join("./datasets/faces/part" + str(dataset+1))
  if (os.path.exists(dataset_path)):
    source_folders.append(dataset_path)

for folder in source_folders:
  file_names = os.listdir(folder)
  for file_name in file_names:
    shutil.move(os.path.join(folder, file_name), faces_images_path)
  os.rmdir(folder) # remove directory because we don't need it anymore

In [ ]:
!pip install pillow rich rich-pixels

In [ ]:
file_names = os.listdir(faces_images_path)
print(f"Number of images: {len(file_names)}")

Cropping Images

In [ ]:
!pip install face-recognition

In [ ]:
import face_recognition
from PIL import Image, ImageOps

os.makedirs(os.path.join(face_path, "temp"), exist_ok=True)
os.makedirs(os.path.join(face_path, "cropped"), exist_ok=True)
test_set = file_names[:500] # test batch

# function to upscale or downscale each image to a consistent size
def resize_image(image_file, target_size=(160,160)):
  with Image.open(image_file) as image_to_process:
    processed_image = ImageOps.pad(
        image_to_process,
        target_size,
        Image.Resampling.LANCZOS,
        (0,0,0) # black padding
    )

    return processed_image

for file in test_set:
  image = face_recognition.load_image_file(os.path.join("/content/datasets/faces/images", file))
  face_locations = face_recognition.face_locations(image, model="cnn")
  if len(face_locations) > 0:
    top, right, bottom, left = face_locations[0]
    width = right - left
    height = bottom - top


    # get cropped image
    face_image = image[top:bottom, left:right]
    pil_image = Image.fromarray(face_image)

    # save crop
    temp_image_location = f"/content/datasets/faces/temp/{file}"
    pil_image.save(temp_image_location)

    # add padding
    ready_image = resize_image(temp_image_location)
    ready_image.save(f"/content/datasets/faces/cropped/{file}")

!rm -rf datasets/faces/temp/*
!rmdir datasets/faces/temp


Creating CSV

In [ ]:
!rm -rf datasets/faces/cropped/*